In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [6]:
class HousingDataset(Dataset):
    """Handles simultaneous loading of house images and tabular features."""
    def __init__(self, tabular_df, image_paths, transform=None):
        self.tabular_data = torch.tensor(tabular_df.drop('price', axis=1).values, dtype=torch.float32)
        self.prices = torch.tensor(tabular_df['price'].values, dtype=torch.float32).view(-1, 1)
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.prices)

    def __getitem__(self, idx):
        # Image Processing (Creating a dummy image for demonstration)
        # In real use, replace with: img = Image.open(self.image_paths[idx]).convert('RGB')
        img = Image.fromarray(np.uint8(np.random.rand(224, 224, 3) * 255))
        
        if self.transform:
            img = self.transform(img)
            
        tabular = self.tabular_data[idx]
        price = self.prices[idx]
        
        return img, tabular, price

In [7]:
class MultimodalHouseModel(nn.Module):
    def __init__(self, num_tabular_features):
        super(MultimodalHouseModel, self).__init__()
        
        self.image_branch = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Output: 112x112
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # Output: 56x56
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 64),
            nn.ReLU()
        )
        
        self.tabular_branch = nn.Sequential(
            nn.Linear(num_tabular_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )
        
        self.regressor = nn.Sequential(
            nn.Linear(64 + 16, 32),
            nn.ReLU(),
            nn.Linear(32, 1) # Predicting Price
        )

    def forward(self, img, tab):
        img_feats = self.image_branch(img)
        tab_feats = self.tabular_branch(tab)
        combined = torch.cat((img_feats, tab_feats), dim=1)
        return self.regressor(combined)

In [8]:
data = pd.DataFrame(np.random.rand(100, 11), columns=[f'feat_{i}' for i in range(10)] + ['price'])
img_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = HousingDataset(data, image_paths=["path"]*100, transform=img_transforms)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

model = MultimodalHouseModel(num_tabular_features=10)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [9]:
print("Starting Training...")
model.train()
for epoch in range(5):
    total_loss = 0
    for imgs, tabs, prices in dataloader:
        optimizer.zero_grad()
        outputs = model(imgs, tabs)
        loss = criterion(outputs, prices)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Average Loss: {total_loss/len(dataloader):.4f}")

Starting Training...
Epoch 1 | Average Loss: 8.1331
Epoch 2 | Average Loss: 0.0985
Epoch 3 | Average Loss: 0.1139
Epoch 4 | Average Loss: 0.0956
Epoch 5 | Average Loss: 0.0995


In [10]:
model.eval()
all_preds, all_actuals = [], []
with torch.no_grad():
    for imgs, tabs, prices in dataloader:
        preds = model(imgs, tabs)
        all_preds.extend(preds.numpy())
        all_actuals.extend(prices.numpy())

mae = mean_absolute_error(all_actuals, all_preds)
rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))

print(f"\nFINAL EVALUATION METRICS:")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")


FINAL EVALUATION METRICS:
Mean Absolute Error (MAE): 0.2676
Root Mean Squared Error (RMSE): 0.3070
